In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action="ignore")

pd.set_option('display.max_columns',1000)
pd.set_option('display.width', 500)
pd.set_option('display.float_format',lambda x : '%.2f' % x)

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df_ = pd.read_csv("data/dataset.csv", compression="gzip")
df = df_.copy()
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1440,45,1485,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,41,Carina's Tofu-Vegetable Kebabs,20,1440,1460,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc..."
2,42,Cabbage Soup,30,20,50,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil..."
3,45,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u..."
4,46,A Jad - Cucumber Pickle,0,25,25,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then..."


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def grab_col_names(dataframe, cat_th=10, car_th=20):

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 16
cat_cols: 0
num_cols: 13
cat_but_car: 3
num_but_cat: 0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def outlier_thresholds(dataframe, col_name, q1=0.01, q3=0.99):
    quartile1= dataframe[col_name].quantile(q1)
    quartile3= dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 -quartile1
    up_limit= quartile3 +1.5 * interquantile_range
    low_limit= quartile1 -1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(df, col)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

check_outlier(df,num_cols)

False

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
df= df.iloc[:,1:]

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import AgglomerativeClustering

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 15
cat_cols: 0
num_cols: 12
cat_but_car: 3
num_but_cat: 0


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
df2=df.copy()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
sc = MinMaxScaler((0, 1))
df2[num_cols] = sc.fit_transform(df2[num_cols])

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
kmeans = KMeans(n_clusters=30, n_init="auto").fit(df2[["TotalTime","Calories","SugarContent"]])

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
clusters_kmeans = kmeans.labels_
clusters_kmeans

array([ 9,  9, 26, ..., 17,  7,  5], dtype=int32)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df["kmeans_cluster"] = clusters_kmeans
df["kmeans_cluster"]= df["kmeans_cluster"] + 1
df.head()

,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,kmeans_cluster
0,Low-Fat Berry Blue Frozen Dessert,1200,45,1485.00,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan...",10
1,Carina's Tofu-Vegetable Kebabs,20,600,1460.00,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc...",10
2,Cabbage Soup,30,20,50.00,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil...",27
3,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80.00,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u...",18
4,A Jad - Cucumber Pickle,0,25,25.00,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then...",6


In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# df.groupby('kmeans_cluster').agg({1: ['count','mean', 'median', 'sum'],
#                                     2: ['count','mean', 'median', 'sum'],
#                                     3: ['count','mean', 'median', 'sum'],
#                                     4: ['count','mean','median', 'sum']})

# === AFTER (edited) ===
df.groupby('kmeans_cluster')[['CookTime', 'PrepTime', 'TotalTime', 'Calories']].agg(['count', 'mean', 'median', 'sum'])

CookTime                         PrepTime                       TotalTime                            Calories                         
                  count   mean  median      sum    count   mean median     sum     count    mean  median        sum    count   mean median        sum
kmeans_cluster                                                                                                                                       
1                  6155  37.22   25.00   229111     6155  20.30  15.00  124921      6155   57.61   45.00  354560.00     6155 460.02 444.20 2831418.00
2                 20264  24.20   15.00   490360    20264  17.04  10.00  345386     20264   41.28   30.00  836594.00    20264 106.31 104.20 2154233.70
3                  3128  55.71   35.00   174264     3128  25.26  15.00   78999      3128   81.25   55.00  254136.00     3128 572.36 525.35 1790356.70
4                   676 620.24  480.00   419283      676 319.49 600.00  215977       676 1570.95 1470.00 1061960.50      676 282.27 231.30  190817.40
5                 18905  34.27   25.00   647924    18905  18.68  15.00  353168     18905   52.97   40.00 1001481.00    18905 424.17 421.70 8019022.00
6                 34005  17.99   10.00   611722    34005  14.52  10.00  493848     34005   32.54   20.00 1106448.00    34005  61.89  61.20 2104402.30
7                 12852  34.86   20.00   447963    12852  19.07  15.00  245092     12852   53.97   40.00  693651.00    12852 578.37 564.80 7433150.70
8                 11778  33.61   20.00   395890    11778  18.47  15.00  217573     11778   52.13   40.00  614000.00    11778 244.90 246.70 2884380.80
9                 14431  26.54   20.00   382946    14431  17.28  15.00  249371     14431   43.85   35.00  632862.00    14431 193.21 189.60 2788176.40
10                  441 643.32 1020.00   283703      441 311.68 360.00  137450       441 1545.75 1470.00  681675.50      441 349.69 285.00  154211.30
11                 9924  38.27   25.00   379799     9924  20.15  15.00  199975      9924   58.50   45.00  580507.00     9924 436.41 428.30 4330972.30
12                 1291 531.81  120.00   686572     1291 355.74 600.00  459264      1291 1425.65 1455.00 1840508.00     1291 237.56 192.00  306693.40
13                 3895  38.69   20.00   150716     3895  22.67  15.00   88283      3895   61.38   40.00  239092.00     3895 887.99 841.10 3458730.50
14                 7615 375.96  360.00  2862923     7615  82.10  15.00  625172      7615  462.89  480.00 3524938.00     7615 281.57 273.80 2144168.20
15                25419  26.15   15.00   664724    25419  17.40  10.00  442169     25419   43.58   30.00 1107865.00    25419 105.65 109.40 2685459.00
16                17258  38.53   25.00   664880    17258  19.68  15.00  339717     17258   58.29   45.00 1005921.00    17258 330.48 325.90 5703398.90
17                 3016  47.40   25.00   142962     3016  22.40  15.00   67552      3016   69.85   45.00  210659.00     3016 820.24 776.70 2473854.70
18                11771  40.86   25.00   480907    11771  19.21  15.00  226085     11771   60.12   40.00  707629.00    11771 273.89 276.80 3224015.00
19                 6821  39.28   25.00   267921     6821  19.72  15.00  134487      6821   59.14   45.00  403363.00     6821 306.38 316.90 2089802.70
20                 9608  41.49   30.00   398616     9608  20.49  15.00  196830      9608   62.08   45.00  596502.00     9608 310.11 316.55 2979494.60
21                26967  30.53   20.00   823264    26967  17.71  15.00  477576     26967   48.29   40.00 1302263.00    26967 367.46 360.70 9909286.30
22                36560  25.75   18.00   941280    36560  16.85  10.00  616044     36560   42.64   30.00 1559000.00    36560 212.98 211.00 7786368.80
23                18490  25.30   15.00   467797    18490  17.32  15.00  320187     18490   42.64   30.00  788474.00    18490 134.88 132.20 2494004.60
24                 2963  58.44   30.00   173152     2963  23.56  15.00   69800      2963   82.04   50.00  243072.00

In [17]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
assert 'kmeans_cluster' in numeric_columns, 'Expected kmeans_cluster to be a numeric grouping column.'
numeric_columns.remove('kmeans_cluster')
assert len(numeric_columns) >= 5, 'Expected at least five numeric feature columns for this aggregation test.'

agg_result = df.groupby('kmeans_cluster').agg({
    numeric_columns[1]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[2]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[3]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[4]: ['count', 'mean', 'median', 'sum'],
})
assert not agg_result.empty, 'Grouped aggregation should produce a non-empty result.'
assert set(['count', 'mean', 'median', 'sum']).issubset(set(agg_result.columns.get_level_values(1)))

try:
    df.groupby('kmeans_cluster').agg({
        1: ['count', 'mean', 'median', 'sum'],
        2: ['count', 'mean', 'median', 'sum'],
        3: ['count', 'mean', 'median', 'sum'],
        4: ['count', 'mean', 'median', 'sum'],
    })
except KeyError:
    pass
else:
    raise AssertionError('Bug regression: integer-labeled aggregation keys unexpectedly succeeded.')